In [1]:
import pandas as pd
import numpy as np

In [2]:
sku = pd.read_excel('DA_BH.xlsx', sheet_name='SKU_Master')
stores = pd.read_excel('DA_BH.xlsx', sheet_name='Stores')
df  = pd.read_csv('sales_clean.csv')

In [3]:
df = df.merge(sku[['sku', 'category', 'unit_cost']], on='sku', how='left')
assert df['category'].notna().all(), "Còn sku chưa chuẩn hóa"


In [4]:
df['cogs'] = df['unit_cost'] * df['qty']
df['is_sale'] = df['qty'] > 0
df.head()

,invoice_id,date,store_id,sku,qty,unit_price,discount_pct,promo_code,net_amount,category,unit_cost,cogs,is_sale
0,HD260700000,2026-07-01,CT01,SKU045,1,613000,0.0,NaN,613000,Áo khoác,270000,270000,True
1,HD260700001,2026-07-01,CT01,SKU038,1,329000,0.0,NaN,329000,Váy,136000,136000,True
2,HD260700002,2026-07-01,CT01,SKU009,2,261000,0.0,NaN,522000,Áo thun,101000,202000,True
3,HD260700003,2026-07-01,CT01,SKU002,3,322000,0.0,NaN,966000,Áo thun,149000,447000,True
4,HD260700004,2026-07-01,CT01,SKU022,2,630000,0.0,NaN,1260000,Quần jeans,263000,526000,True


In [5]:
def kpi(g):
    sale = g[g['qty'] > 0]
    inv = g['invoice_id'].unique()
    
    return {
        'doanh thu': sale['net_amount'].sum(),
        'lợi nhuận gộp': sale['net_amount'].sum() - sale['cogs'].sum(),
        'số hóa đơn': len(inv),
        'số sản phẩm bán ra': sale['qty'].sum(),
        'số sản phẩm trả lại': -g[g['qty'] < 0]['qty'].sum(),
        'tỉ lệ trả hàng': -g[g['qty'] < 0]['qty'].sum() / sale['qty'].sum() if sale['qty'].sum() > 0 else 0,
        'UPT': sale['qty'].sum() / len(inv) if len(inv) > 0 else 0,
        'ATV': sale['net_amount'].sum() / len(inv) if len(inv) > 0 else 0,
        'tỉ suất lợi nhuận gộp': 1 - g['cogs'].sum() / g['net_amount'].sum() if g['net_amount'].sum() > 0 else 0
    }
    
    

In [6]:
#bảng theo cửa  hàng
by_store = (df[df['store_id'] != 'UNKNOWN']).groupby('store_id').apply(kpi, include_groups = False)

#bảng theo ngành hàng
by_cat = (df[df['store_id'] != 'UNKNOWN']).groupby('category').apply(kpi, include_groups = False)

by_store.head()


store_id
CT01     {'doanh thu': 189082800, 'lợi nhuận gộp': 1059...
DN01     {'doanh thu': 302505200, 'lợi nhuận gộp': 1690...
HCM01    {'doanh thu': 662427200, 'lợi nhuận gộp': 3701...
HCM02    {'doanh thu': 451866000, 'lợi nhuận gộp': 2523...
HN01     {'doanh thu': 552072300, 'lợi nhuận gộp': 3084...
dtype: object

In [15]:
stores["open_date"] = pd.to_datetime(stores["open_date"])

q3_start = pd.Timestamp("2026-07-01")
q3_end = pd.Timestamp("2026-09-30")

# by_store là Series sau groupby(...)[...].sum()
revenue_by_store = (
    by_store
    .rename("doanh thu")
    .reset_index()
)

m2 = stores.merge(
    revenue_by_store,
    on="store_id",
    how="left"
)

# Chỉ tính các cửa hàng đã mở trước hoặc trong quý 3
m2["days_open"] = (
    q3_end - m2["open_date"].clip(lower=q3_start, upper=q3_end)
).dt.days + 1

m2["months_open"] = (
    (q3_end - m2["open_date"]).dt.days / 30.4
).round(1)


m2["rev_per_m2"] = m2["doanh thu"] / m2["area_m2"]
m2["rev_per_m2_day"] = m2["rev_per_m2"] / m2["days_open"]
m2["ramp_up"] = m2["months_open"] < 6

# Xếp hạng chỉ trong nhóm cửa hàng đã ổn định
mature = m2.loc[
    (~m2["ramp_up"]) &
    (m2["rev_per_m2_day"].notna()) &
    (m2["rev_per_m2_day"].notnull())
].copy()

mature["rank"] = (
    mature["rev_per_m2_day"]
    .rank(ascending=False, method="min")
    .astype("Int64")
)

print(m2.sort_values("rev_per_m2_day", ascending=False).round(0))

TypeError: unsupported operand type(s) for /: 'dict' and 'int'

In [11]:
print(by_store.columns.tolist())
print(by_store.head(3))

AttributeError: 'Series' object has no attribute 'columns'